### Load all simulation complexes

Load all simulation complexes.


In [1]:
import os
import glob
from tqdm.notebook import tqdm
import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt

homedir = os.getenv("HOME")
gdrive_mnt = "mnt/gdrive"
datadir = 'data/GROMACS'
datadir_path = os.path.join(homedir, gdrive_mnt, datadir)

# Find all simulation directories starting with cuedc2_
simulation_dirs = [
    os.path.join(datadir_path, entry)
    for entry in sorted(os.listdir(datadir_path))
    if entry.startswith("cuedc2_") and os.path.isdir(os.path.join(datadir_path, entry))
]


In [4]:
simulation_dirs = ["/home/daneel/mnt/gdrive/data/GROMACS/cuedc2_erg"]

Prepare protlig tpr selections for each complex (unless it already exists from old PP)

Now, prepare MDAnalysis universes for each complex

In [ ]:
complexes = {}

for dirpath in tqdm(simulation_dirs):
    complex_name = os.path.basename(dirpath)
    
    # Recursively find traj_dry.tpr and *_compact_compact.xtc
    tpr_files = glob.glob(os.path.join(dirpath, "**", "traj_dry.tpr"), recursive=True)
    xtc_files = sorted(glob.glob(os.path.join(dirpath, "**", "*_compact_compact.xtc"), recursive=True))
    
    if not tpr_files:
        print(f"Skipping {complex_name}: 'traj_dry.tpr' not found")
        continue
    if not xtc_files:
        print(f"Skipping {complex_name}: '*_compact_compact.xtc' not found")
        continue
        
    complexes[complex_name] = {
        "tpr": tpr_files[0],
        "xtc": xtc_files[0] if len(xtc_files) == 1 else xtc_files,
    }
    print(f"Loaded {complex_name}: {len(xtc_files)} trajectory file(s)")


In [ ]:
complexes

### Calculate all RMSD values and load in a dataframe

In [ ]:
from multiprocessing import Pool, cpu_count
import os
import pandas as pd

def _compute_rmsd_task(args):
    complexname, topol_path, traj_path = args
    import MDAnalysis as mda
    from MDAnalysis.analysis import rms
    u = mda.Universe(topol_path, traj_path)
    R = rms.RMSD(u, u, select='backbone', ref_frame=0)
    R.run(verbose=False)
    times = R.results.rmsd[:, 1]
    rmsd = R.results.rmsd[:, 2]
    return complexname, times, rmsd

# Build list of (name, topol_path, traj_path) so worker processes don't need non-picklable Universe objects
tasks = [
    (complex_name, data["tpr"], data["xtc"])
    for complex_name, data in complexes.items()
]

rmsd_df = None
# If SLURM_CPUS_PER_TASK is set, use it (fall back to previously computed nproc)
slurm_cpus = 1 #os.environ.get('SLURM_CPUS_PER_TASK')
if slurm_cpus:
    try:
        nproc_env = int(slurm_cpus)
        if nproc_env > 0:
            nproc = nproc_env
    except ValueError:
        pass

# Ensure we don't request more processes than logical CPUs
nproc = max(1, min(nproc, cpu_count()))

print(f"Using nproc={nproc}")
with Pool(processes=nproc) as pool:
    for complexname, times, rmsd in tqdm(pool.imap_unordered(_compute_rmsd_task, tasks), total=len(tasks)):
        if rmsd_df is None:
            rmsd_df = pd.DataFrame({'Time': times})
        rmsd_df[complexname] = rmsd

# Prepare and display the plot.
rmsd_df.set_index('Time', inplace=True)
rmsd_df.plot(legend=False)
plt.xlabel('Time (ps)')
plt.ylabel(r'RMSD ($\AA$)')
plt.title('RMSD of all complexes')
#plt.legend(title='Complex')
plt.show()


In [ ]:
rmsd_df.head()

### Calculate all RMSF and load in a dictionary

In [ ]:
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import numpy as _np

# Build list of (name, topol_path, traj_path) so worker processes don't need non-picklable Universe objects
tasks = [
    (complex_name, data["tpr"], data["xtc"])
    for complex_name, data in complexes.items()
]

# worker (override previous broken definition)
def _compute_rmsf_task(args):
    complexname, topol_path, traj_path = args

    u = mda.Universe(topol_path, traj_path)
    sel = u.select_atoms('backbone and name CA')
    R = rms.RMSF(sel).run(verbose=False)
    resids = _np.array(sel.resids)
    rmsf = _np.array(R.results.rmsf)
    return complexname, resids, rmsf

# run in parallel and collect results
rmsf_dict = {}
with Pool(processes=nproc) as p:
    for complexname, resids, rmsf in tqdm(p.imap_unordered(_compute_rmsf_task, tasks), total=len(tasks)):
        rmsf_dict[f'{complexname}_resid'] = resids
        rmsf_dict[f'{complexname}_rmsf'] = rmsf

### Calculate all Radii of Gyration and load in a dataframe 

In [ ]:
import pandas as pd
from MDAnalysis.analysis import rms
import MDAnalysis as mda
import numpy as _np
# Build list of (name, topol_path, traj_path) so worker processes don't need non-picklable Universe objects
rg_tasks = [
    (complex_name, data["tpr"], data["xtc"])
    for complex_name, data in complexes.items()
]

def _compute_rg_task(args):
    complexname, topol_path, traj_path = args
    u = mda.Universe(topol_path, traj_path)
    group = u.select_atoms('protein')
    times = []
    rg = []
    for ts in u.trajectory:
        times.append(ts.time)
        rg.append(group.radius_of_gyration())
    return complexname, _np.array(times), _np.array(rg)

rg_df = None
with Pool(processes=nproc) as pool:
    for complexname, times_arr, rg_arr in tqdm(pool.imap_unordered(_compute_rg_task, rg_tasks), total=len(rg_tasks)):
        if rg_df is None:
            rg_df = pd.DataFrame({'Time': times_arr})
        rg_df[complexname] = rg_arr

# Prepare and display the plot.
rg_df.set_index('Time', inplace=True)
rg_df.plot(legend=False)
plt.xlabel('Time (ps)')
plt.ylabel(r'$R_g\; (\AA)$')
plt.title('Rg of all complexes')
plt.show()

### Number of hydrogen bonds between protein and ligand

In [ ]:
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA
import numpy as np
import pandas as pd
import os
from multiprocessing import Pool
import numpy as _np
import MDAnalysis as mda
import warnings
import logging
warnings.filterwarnings("ignore")

logging.getLogger("MDAnalysis").setLevel(logging.ERROR)
logging.getLogger("matplotlib").setLevel(logging.ERROR)

# suppress numpy runtime warnings (e.g. divide by zero, invalid)
_np.seterr(all='ignore')
hb_dict = {}
# build tasks: (complexname, topol_path, traj_path)
# Build list of (name, topol_path, traj_path) so worker processes don't need non-picklable Universe objects
tasks_hb = [
    (complex_name, data["tpr"], data["xtc"])
    for complex_name, data in complexes.items()
]

# Temporarily prevent the serial loop below from executing by emptying complexes,
# we'll restore it after the parallel work.
_orig_complexes = complexes
complexes = {}

def _hba_worker(args):
    complexname, topol_path, traj_path = args

    u = mda.Universe(topol_path, traj_path)
    hbs = HBA(universe=u, update_selections=False)
    hbs.donors_sel = hbs.guess_donors("protein")
    hbs.hydrogens_sel = hbs.guess_hydrogens("protein")
    hbs.acceptors_sel = hbs.guess_acceptors("not protein")
    hbs.run(verbose=False)
    ligand_indices = u.select_atoms(hbs.acceptors_sel).indices

    hbs_reversed = HBA(universe=u, update_selections=False)
    hbs_reversed.donors_sel = hbs.guess_donors("not protein")
    hbs_reversed.hydrogens_sel = hbs.guess_hydrogens("not protein")
    hbs_reversed.acceptors_sel = hbs.guess_acceptors("protein")
    hbs_reversed.run(verbose=False)

    # filter to intermolecular bonds (protein <-> ligand)
    filtered = []
    for hb in hbs.results.hbonds:
        frame, donor_idx, hydrogen_idx, acceptor_idx, da_dist, da_angle = hb
        if (donor_idx not in ligand_indices and acceptor_idx in ligand_indices) or \
        (donor_idx in ligand_indices and acceptor_idx not in ligand_indices):
            filtered.append(hb)

    for hb in hbs_reversed.results.hbonds:
        frame, donor_idx, hydrogen_idx, acceptor_idx, da_dist, da_angle = hb
        if (donor_idx not in ligand_indices and acceptor_idx in ligand_indices) or \
        (donor_idx in ligand_indices and acceptor_idx not in ligand_indices):
            filtered.append(hb)

    filtered = np.array(filtered)

    # counts per frame (nhb)
    times = hbs.times
    if filtered.size == 0:
        nhb = _np.zeros_like(times, dtype=int)
    else:
        frames = filtered[:, 0].astype(int)
        nhb = _np.bincount(frames, minlength=len(times))[:len(times)]

    n_frames = hbs.n_frames

    # counts by (donor, hydrogen, acceptor)
    if filtered.size == 0:
        count_ids = _np.empty((0, 4), dtype=int)
    else:
        ids = filtered[:, 1:4].astype(int)
        uniq, counts = _np.unique(ids, axis=0, return_counts=True)
        count_ids = _np.hstack((uniq, counts.reshape(-1, 1))).astype(int)

    return complexname, {
        "hbonds": filtered,
        "times": times,
        "nhb": nhb,
        "n_frames": n_frames,
        "count_ids": count_ids,
    }

# Run workers and build hb_dict with lightweight stubs compatible with later code
hb_dict = {}
with Pool(processes=nproc) as pool:
    for complexname, data in pool.imap_unordered(_hba_worker, tasks_hb):
        class _HBAStub:
            def __init__(self, d):
                self.results = type("R", (), {})()
                self.results.hbonds = d["hbonds"]
                self.times = d["times"]
                self.n_frames = d["n_frames"]
                self._nhb = d["nhb"]
                self._count_ids = d["count_ids"]
            def count_by_time(self):
                return self._nhb
            def count_by_ids(self):
                return self._count_ids

        hb_dict[complexname] = _HBAStub(data)
# Restore complexes dictionary for potential later use
complexes = _orig_complexes
nhb_df = None
for complexname, hbs in hb_dict.items():
    times = hbs.times
    nhb = hbs.count_by_time()
    # For the first complex, initialize the DataFrame with the time column.
    if nhb_df is None:
        nhb_df = pd.DataFrame({'Time': times})
        
    # Add the RMSD values for the current complex to the DataFrame.
    nhb_df[complexname] = nhb
# Prepare and display the plot.
# Prepare and display the plot.
nhb_df.set_index('Time', inplace=True)
nhb_df.plot(legend=False)
plt.xlabel('Time (ps)')
plt.ylabel('Hnb (nm)')
plt.title('Nhb of all complexes')
plt.show()

In [ ]:
from IPython.display import display

# find and show columns in nhb_df that contain only zeros
try:
    nhb_df  # ensure variable exists
except NameError:
    print("nhb_df is not defined in the current notebook.")
else:
    zero_cols = [c for c in nhb_df.columns if nhb_df[c].eq(0).all()]
    if not zero_cols:
        print("No columns contain only zeros.")
    else:
        print(f"Columns with all zeros ({len(zero_cols)}):")
        for name in zero_cols:
            print(" -", name)
        # display the data for those columns (useful in Jupyter)
        try:
            display(nhb_df[zero_cols])
        except Exception:
            print(nhb_df[zero_cols].head())

### Plot everything

### Hydrogen Bond occupancy statistics

In [ ]:
from tqdm.notebook import tqdm
hb_details = {}

for complex, hb in tqdm(hb_dict.items()):
    u = complexes[complex]
    ligand_indices = u.select_atoms('not protein').indices
    ligand_names = u.select_atoms('not protein').resnames
    hbstats = hb.count_by_ids()
    # Only keep bonds with occupancy more than 0.1%
    hbstats_filtered = hbstats[hbstats[:, -1] / hb.n_frames > 0.001]
    bond_types = []
    for bond in hbstats_filtered:
        donor_idx, hydrogen_idx, acceptor_idx, occ = bond
        donor_atom = u.atoms[int(donor_idx)]
        hydrogen_atom = u.atoms[int(hydrogen_idx)]
        acceptor_atom = u.atoms[int(acceptor_idx)]
        # Check if the donor or acceptor is a ligand
        if donor_idx in ligand_indices and acceptor_idx not in ligand_indices:
            # Donor is a ligand, acceptor is not
            bond_type = f"({donor_atom.resname}){donor_atom.name}-H::{acceptor_atom.name}({acceptor_atom.resid}-{acceptor_atom.resname}) : {occ/hb.n_frames:.3f}"
        elif acceptor_idx in ligand_indices and donor_idx not in ligand_indices:
            # Acceptor is a ligand, donor is not
            bond_type = f"({donor_atom.resid}-{donor_atom.resname}){donor_atom.name}-H::{acceptor_atom.name}({acceptor_atom.resname}) : {occ/hb.n_frames:.3f}"
        else:
            # Both donor and acceptor are ligands
            continue
        bond_types.append(bond_type)
    
    hb_details[complex] = bond_types

### Save the hydrogen bond data to a json file so you don't have to keep calculating it

In [ ]:
import json

outpath = os.path.join(data_dir, "hb_details.json")
with open(outpath, "w") as fh:
    json.dump(hb_details, fh, indent=2)
print(f"Saved hb_details to {outpath}")

### Load the hbonds info from json file to plot

## Plot everything